# SIFLOW · run_6 · Dream-7B train + eval (SIFLOW-D)

Head-only SIFLOW-D on the Dream-7B backbone (live, reduced top-m support), then eval. ~6–8h. Fills the SIFLOW-D rows of Table 2.

**Runtime:** designed to finish well under one Colab session. Training stops and checkpoints automatically at an 11h guard — if that happens, just re-run this notebook (re-import its checkpoint) and it resumes.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

In [ ]:
# === Hugging Face login ===
# Required for the gated DiffusionGemma weights; recommended for Dream too.
# Get a token at https://huggingface.co/settings/tokens (read scope).
from huggingface_hub import login
login()

### Import the previous part(s)

This part needs the output zip(s) you downloaded earlier. Run the cell below — a file picker opens; select **all** of these at once:

- `run_5_dream_data.zip` — Dream-tokenized data
- `run_3_results.zip / run_4_ablations.zip` — (optional) to keep prior rows in the table

*(If a long run here stopped early at the 11h guard, also re-upload **this** part's own checkpoint zip to resume.)* Using Drive instead? Set `USE_DRIVE=True` above and skip this.

In [ ]:
# === Import previous outputs (pick the .zip files listed above) ===
if not USE_DRIVE:
    from siflow.utils.io import import_zips
    import_zips(BASE)
else:
    print('USE_DRIVE: reading prior outputs from', BASE)

In [ ]:
!python scripts/train.py --config siflow/config/dream.yaml --set \
    data.tokens_path={BASE}/data/dream_256.npy \
    out_dir={BASE}/ckpt/dream run_id=siflow_dream train.total_steps=12000

In [ ]:
!python scripts/evaluate.py --config siflow/config/dream.yaml --system siflow \
    --ckpt-dir {BASE}/ckpt/dream --ref-tokens {BASE}/data/dream_val.npy \
    --n-samples 500 --k-list 1 4 --out {BASE}/results/dream.json

In [ ]:
!python scripts/make_tables.py --results {BASE}/results --out {BASE}/tables_auto.tex

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'run_6_dream.zip', include=['ckpt/dream', 'results/dream.json'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)